# Week 2 · Class 4 — Data Collection for RAG

Scrape real web data, parse a PDF, chunk the text, and feed a real chunk into the `docs` node you
built in Class 3.

## Before you start

This notebook is standalone and runs top to bottom on its own. Section 3 quickly rebuilds just the
one piece of your Class 3 agent we need — the context-aware `docs` chain — so the final wiring
step actually runs here without repeating the full LangGraph build.

Get a free API key at [console.groq.com](https://console.groq.com) before Section 2.


## Section 1 — Install packages


In [ ]:
!pip install -q langchain-core langchain-groq requests beautifulsoup4 pandas pymupdf


## Section 2 — Groq API key


In [ ]:
import os
import getpass

# Try Colab Secrets first; fall back to a prompt.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))


## Section 3 — Recap: the `docs` chain from Class 3

This is the same prompt and logic as the `docs_node` you built in Class 3 — condensed to a single
function so this notebook can run on its own. If you already have your Class 3 agent open in
another tab, you can skip straight to Section 4 and come back to wire data into your real
`agent` there instead.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL, temperature=0.3)

docs_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the reference text below. "
               "If the answer isn't in the text, say you don't have that information."),
    ("human", "Reference text:\n{context}\n\nQuestion: {question}"),
])

def answer_from_context(question: str, context: str) -> str:
    result = (docs_prompt | llm).invoke({"question": question, "context": context})
    return result.content

# Quick sanity check with made-up context
print(answer_from_context(
    "What does Groq build?",
    "Groq is a company that builds fast AI inference hardware called LPUs.",
))


## Section 4 — Demo 1: scrape a blog you can see

Open [Python Insider](https://blog.python.org/) in another tab. Before running the code, ask the
class to point out the repeated **title, author, date, and summary** on each post card. We will
turn exactly those visible cards into rows of data.


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

HEADERS = {"User-Agent": "GlobalChatbotBootcamp/1.0 (educational demo)"}
BLOG_URL = "https://blog.python.org/"

response = requests.get(BLOG_URL, headers=HEADERS, timeout=20)
response.raise_for_status()
blog_soup = BeautifulSoup(response.text, "html.parser")

print("Status:", response.status_code)
print("Repeated <article> cards found:", len(blog_soup.select("article")))


## Section 5 — Extract and display the blog posts

CSS selectors describe what we can already see: `article` finds each repeated card, then
`h3`, `time`, and `p` find fields inside that card.


In [ ]:
blog_posts = []
for card in blog_soup.select("article"):
    title_el = card.select_one("h3")
    link_el = card.select_one("a[href]")
    date_el = card.select_one("time")
    summary_el = card.select_one("p")
    author_el = date_el.parent.select_one("span") if date_el else None

    if title_el and link_el:
        blog_posts.append({
            "title": title_el.get_text(" ", strip=True),
            "author": author_el.get_text(" ", strip=True) if author_el else "Unknown",
            "date": date_el.get("datetime", date_el.get_text(strip=True)) if date_el else "",
            "summary": summary_el.get_text(" ", strip=True) if summary_el else "",
            "url": urljoin(BLOG_URL, link_el["href"]),
        })

assert blog_posts, "No blog cards found — inspect the page because its HTML may have changed."
print(f"Scraped {len(blog_posts)} posts")
display(pd.DataFrame(blog_posts)[["title", "author", "date", "summary"]])


## Section 6 — Scrape responsibly

- Check the site's terms and `robots.txt`; permission to view is not always permission to crawl
- Identify your client with a truthful `User-Agent`
- Cache responses and add a delay between repeated requests
- Never collect private, copyrighted, or personal data without a legitimate reason
- Prefer an official API when one exists


In [ ]:
# Save the first demo while the data is still small and easy to inspect.
import json
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

blog_path = data_dir / "python_blog_posts.json"
with open(blog_path, "w", encoding="utf-8") as f:
    json.dump(blog_posts, f, indent=2, ensure_ascii=False)
print(f"Saved {len(blog_posts)} posts to {blog_path}")


## Section 7 — Demo 2: 2022 World Cup group tables

Now move from repeated cards to nested tables. Open the
[2022 FIFA World Cup group stage](https://en.wikipedia.org/wiki/2022_FIFA_World_Cup#Group_stage)
and compare the page with the rows printed below. The historical page is used so results do not
change while students are learning.


In [ ]:
WORLD_CUP_URL = "https://en.wikipedia.org/wiki/2022_FIFA_World_Cup"
wc_response = requests.get(WORLD_CUP_URL, headers=HEADERS, timeout=30)
wc_response.raise_for_status()
wc_soup = BeautifulSoup(wc_response.text, "html.parser")

print("Status:", wc_response.status_code)
print("Group headings found:", [wc_soup.find(id=f"Group_{g}").get_text(strip=True) for g in "ABCDEFGH"])


## Section 8 — Turn eight HTML tables into one dataset

Instead of depending on a table's position on the page, find each group heading by ID and then
find the next standings table. This is more explainable and less brittle than "table number 12."


In [ ]:
standings = []
for group in "ABCDEFGH":
    heading = wc_soup.find(id=f"Group_{group}")
    table = heading.find_next("table", class_="wikitable")

    for row in table.select("tr")[1:]:  # skip the header row
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) < 10:
            continue
        team_link = cells[1].find("a", title=lambda t: t and t.endswith("national football team"))
        team = team_link.get_text(" ", strip=True) if team_link else cells[1].get_text(" ", strip=True)
        standings.append({
            "group": group,
            "position": int(cells[0].get_text(strip=True)),
            "team": team,
            "played": int(cells[2].get_text(strip=True)),
            "won": int(cells[3].get_text(strip=True)),
            "drawn": int(cells[4].get_text(strip=True)),
            "lost": int(cells[5].get_text(strip=True)),
            "goals_for": int(cells[6].get_text(strip=True)),
            "goals_against": int(cells[7].get_text(strip=True)),
            "goal_difference": int(cells[8].get_text(strip=True).replace("−", "-")),
            "points": int(cells[9].get_text(strip=True)),
        })

assert len(standings) == 32, f"Expected 32 teams, found {len(standings)}"
standings_df = pd.DataFrame(standings)
display(standings_df)

standings_path = data_dir / "world_cup_2022_group_standings.json"
with open(standings_path, "w", encoding="utf-8") as f:
    json.dump(standings, f, indent=2, ensure_ascii=False)
print(f"Saved {len(standings)} teams to {standings_path}")


## Section 9 — Student challenge: build a tiny book crawler

**Your mission:** crawl the first three pages of [Books to Scrape](https://books.toscrape.com/),
then visit every book's detail page. Collect `title`, `category`, numeric `price_gbp`, star
`rating`, and `availability`, deduplicate by URL, and save `data/scraped_books_enriched.json`.

Why it is harder: one record now combines data from a listing page and a detail page, and links
are relative to the **current page**. Never build the next URL by string concatenation. The
starter below deliberately completes only the listing-page discovery step.

Success checks: 60 unique books from 3 pages, no missing fields, numeric prices, and at least a
one-second delay between page requests. Bonus: calculate average price by category.


In [ ]:
BOOKS_URL = "https://books.toscrape.com/"

def discover_book_links(page_url: str):
    response = requests.get(page_url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    page_soup = BeautifulSoup(response.text, "html.parser")

    book_urls = [
        urljoin(page_url, a["href"])
        for a in page_soup.select("article.product_pod h3 a[href]")
    ]
    next_link = page_soup.select_one("li.next a[href]")
    next_url = urljoin(page_url, next_link["href"]) if next_link else None
    return book_urls, next_url

first_page_books, next_page = discover_book_links(BOOKS_URL)
print(f"Discovered {len(first_page_books)} detail pages")
print("First detail URL:", first_page_books[0])
print("Correct next URL:", next_page)

# TODO 1: write scrape_book_details(book_url)
# TODO 2: loop through 3 listing pages and their detail links (use time.sleep)
# TODO 3: validate, deduplicate, and save the enriched records


## Section 10 — PDF text extraction

Use the sample PDF in `week-2/data/sample.pdf` or upload your own document.


In [ ]:
import fitz  # PyMuPDF

pdf_path = Path("data/sample.pdf")
if not pdf_path.exists():
    alt = Path("week-2/data/sample.pdf")
    pdf_path = alt if alt.exists() else pdf_path

if pdf_path.exists():
    doc = fitz.open(pdf_path)
    full_text = "".join(page.get_text() for page in doc)
    print(f"Extracted {len(full_text)} characters from {doc.page_count} pages")
    print(full_text[:400])
else:
    print("Upload a PDF to data/sample.pdf or set pdf_path to your file.")
    full_text = ""  # replace after uploading


## Section 11 — Alternative: pdfplumber

Useful for PDFs with tables.


In [ ]:
# !pip install -q pdfplumber
# import pdfplumber
# with pdfplumber.open("data/sample.pdf") as pdf:
#     full_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
# print(full_text[:400])


## Section 12 — Chunking for embeddings


In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        start = end - overlap
    return chunks

chunks = chunk_text(full_text) if full_text else []
print(f"Created {len(chunks)} chunks")
if chunks:
    print("First chunk:", chunks[0][:200], "...")

chunks_path = data_dir / "chunks.json"
with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print(f"Saved to {chunks_path}")


## Section 13 — Wire real data into `docs`

This is the payoff: pass a real scraped/PDF chunk as context, and the answer is grounded in
**your** data instead of the model's memory. If you have your Class 3 `agent` running elsewhere,
replace `answer_from_context(...)` with `agent.invoke({"messages": [...], "context_chunks": ...})`
using that real graph.


In [ ]:
sample_chunk = (
    chunks[0]
    if chunks
    else blog_posts[0]["summary"]
    if blog_posts
    else "No data collected yet."
)

answer = answer_from_context("Summarize this text in one sentence.", sample_chunk)
print("Context used:", sample_chunk[:150], "...")
print("\nAnswer:", answer)


## Section 14 — Deliverable checklist

- [ ] `data/python_blog_posts.json` saved
- [ ] `data/world_cup_2022_group_standings.json` contains 32 teams
- [ ] Challenge: `data/scraped_books_enriched.json` contains 60 unique enriched books
- [ ] At least one PDF parsed and chunked to `data/chunks.json`
- [ ] `docs` (or your Class 3 `agent`'s `docs` node) answers using a real chunk from your dataset
- [ ] Demo ready (2–3 min): show your Class 3 agent's four intents + one scrape sample + one PDF
      chunk feeding a real, grounded answer

**Week 3:** your chunks become embeddings in FAISS / Qdrant.


In [ ]:
# Final self-check — uncomment and run when done
# assert Path("data/python_blog_posts.json").exists(), "Missing python_blog_posts.json"
# assert len(standings) == 32, "Expected all 32 World Cup teams"
# assert Path("data/chunks.json").exists(), "Missing chunks.json"
# print("Deliverable files present. Ready for Week 3!")
